# Day 30: Document Loading and Chunking

On Day 29 we wrote the facts **by hand**, one per line. A real company does not have that. It has **documents**: a leave policy, working hours, laptop rules...

Today we **load** the company documents with Python and **cut** them into small parts (**chunks**). We learn **all the main ways to cut**, and **how to pick the right one**. On Day 31 the chunks go into Chroma.

## 🎯 The example we will build

```
company_docs/                          chunks
  leave_policy.txt      ──►   leave_policy.txt-0     "Every employee gets 20 days of paid leave ..."
  working_hours.txt           leave_policy.txt-1     "To take leave, send a request ..."
  laptop_rules.txt            working_hours.txt-0    "The office is open from 9 am ..."
  expense_policy.txt          ...
```

No API today. Only Python.

## 📋 Agenda

```
TOP     Step 5   All documents → chunks     the list for Day 31
          ▲
        Step 4   How to pick a way          a simple guide
          ▲
        Step 3   The ways to chunk          fixed, sentence, paragraph, recursive, semantic ...
          ▲
        Step 2   Why chunk?                 one vector per idea
          ▲
BOTTOM  Step 1   Load the documents         read the .txt files
```

### New words today

| Word | Meaning |
|---|---|
| **Document** | One file with text, like `leave_policy.txt`. |
| **Load** | Read a file into Python. |
| **Chunk** | A small part of a document. |
| **Chunking** | Cutting documents into chunks. |
| **Overlap** | The end of one chunk is repeated at the start of the next one. |
| **Metadata** | Extra facts about a chunk: file name, heading, date. |
| **Source** | The file a chunk came from. We need it on Day 32 to show where an answer came from. |

---
# Class Problem: cut the company documents into chunks

## Step 1: Load the documents

The folder `company_docs` has 4 text files. `Path` (from `pathlib`) helps us work with files and folders.

In [30]:
from pathlib import Path

for path in sorted(Path("company_docs").glob("*.txt")):      # every .txt file in the folder
    print(path.name)

expense_policy.txt
laptop_rules.txt
leave_policy.txt
working_hours.txt


In [31]:
text = Path("company_docs/laptop_rules.txt").read_text(encoding="utf-8")    # read one file
print(text)

Laptop Rules

Every new employee gets one company laptop on the first day. You must return it on your last day.

Do not install unknown software on the company laptop. Ask the IT team first.

If your laptop is lost or stolen, tell the IT team within 24 hours. The IT team will lock the laptop from far away.

Change your laptop password every 90 days. Never share your password with anyone.



Now we load **all** the files. Each document is a small dictionary: the **source** (file name) and the **text**.

In [32]:
def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):
        docs.append({"source": path.name, "text": path.read_text(encoding="utf-8")})
    return docs

docs = load_documents("company_docs")
for doc in docs:
    print(f"{doc['source']:22} {len(doc['text'].split())} words")

expense_policy.txt     68 words
laptop_rules.txt       72 words
leave_policy.txt       74 words
working_hours.txt      86 words


**✏️ Quick Check:** How many **characters** are in `leave_policy.txt`? Use `len(...)` on its text.

In [33]:
# your code here

**Answer:** 373 characters. Find the file by its source, because the list is in A–Z order.

In [34]:
for doc in docs:
    if doc["source"] == "leave_policy.txt":
        print(len(doc["text"]), "characters")

373 characters


## Step 2: Why chunk?

Why not put the **whole** `leave_policy.txt` in Chroma as one item?

1. **One vector for many ideas is blurry.** The file talks about paid leave, how to ask for leave, and sick leave. Its one vector is a **mix**, so a question about sick days does not match it well.
2. **Too many tokens.** A question about sick days does not need the whole file. We want to send only the part that helps (Day 24: tokens = cost).

But a chunk must not be **too small** either. *"It is 10 days."* What is 10 days? The meaning is lost.

*Everyday example:* a student asks one question about a textbook. You do not photocopy the **whole book**, and you do not cut out **one word**. You copy the **one paragraph** that answers it.

**Goal:** one chunk = **one idea**, complete enough to read alone.

## Step 3: The ways to chunk

There are many ways to cut a document. Here is the full list. We try the first ones in code and explain the rest with small examples.

| # | Way | How it cuts | Cost |
|---|---|---|---|
| 1 | **Fixed-size** | Every N characters or tokens | Very cheap |
| 2 | **Sliding window** | Fixed-size, but chunks **overlap** | Cheap |
| 3 | **Sentence-based** | At the end of each sentence | Cheap |
| 4 | **Paragraph-based** | At each blank line | Cheap |
| 5 | **Recursive** | Try big cuts first, then smaller cuts, until it fits | Cheap |
| 6 | **Semantic** | Where the **meaning** changes (uses embeddings) | Medium |
| 7 | **Hierarchical** | Parent (section) and child (paragraph) chunks | Medium |
| 8 | **Contextual** | Add heading, source or date to each chunk | Cheap |
| 9 | **Topic-based** | Group text by topic | Medium |
| 10 | **Modality-specific** | A different way for text, tables and images | Medium |
| 11 | **AI-driven** | An LLM chooses where to cut | Expensive |
| 12 | **Agentic** | An AI agent picks a way for each part of the document | Very expensive |

### Way 1: Fixed-size chunking

**Concept:** cut every N characters (or words, or tokens). It does not look at the meaning.

**Good:** very fast and simple. **Bad:** it cuts words and sentences in the middle.

**Use it for:** a quick first try, or big piles of plain text where speed matters more than quality (like old log files).

In [35]:
def chunk_by_size(text, size=100, overlap=0):
    chunks = []
    step = size - overlap                       # how far we move for the next chunk
    for start in range(0, len(text), step):
        chunks.append(text[start:start + size])
    return chunks

leave = Path("company_docs/leave_policy.txt").read_text(encoding="utf-8")

for chunk in chunk_by_size(leave, size=100)[:4]:
    print(repr(chunk))

'Leave Policy\n\nEvery employee gets 20 days of paid leave in one year. Unused leave days do not carry '
'over to the next year.\n\nTo take leave, send a request to your manager at least 5 days before. The ma'
'nager must reply within 2 days.\n\nSick leave is separate from paid leave. An employee gets 10 sick da'
'ys in one year. After 2 sick days in a row, you must show a doctor note.\n'


**What breaks:** words are cut in two, and one idea is split across two chunks. Look at the end of each chunk.

**✏️ Quick Check:** Cut `leave` with `size=50`. How many chunks do you get? Is it better or worse?

In [36]:
# your code here

**Answer:** 8 chunks. **Worse:** even more sentences are cut in the middle.

In [37]:
small = chunk_by_size(leave, size=50)
print(len(small), "chunks")

8 chunks


### Way 2: Sliding window (overlap)

**Concept:** like fixed-size, but each chunk **shares** some text with the next one. A sentence cut at the end of one chunk is complete in the next.

```
size=100, overlap=20:
chunk 1:  [0 ............................. 100]
chunk 2:                       [80 ............................. 180]
                               └─ 20 shared ─┘
```

**Good:** keeps the context at the edges. **Bad:** more chunks, so more vectors and more cost. A common size is **200–400 tokens** with **20–50% overlap**.

**Use it for:** text with no clear parts, like chat logs or a speech transcript.

In [38]:
for chunk in chunk_by_size(leave, size=100, overlap=20)[:3]:
    print(repr(chunk))

'Leave Policy\n\nEvery employee gets 20 days of paid leave in one year. Unused leave days do not carry '
'e days do not carry over to the next year.\n\nTo take leave, send a request to your manager at least 5'
'r manager at least 5 days before. The manager must reply within 2 days.\n\nSick leave is separate from'


**✏️ Quick Check:** Does `overlap=20` give more or fewer chunks than `overlap=0` (with `size=100`)?

In [39]:
# your code here

**Answer:** more. 4 chunks with no overlap, 5 chunks with `overlap=20`.

In [40]:
print(len(chunk_by_size(leave, 100)), "chunks, no overlap")
print(len(chunk_by_size(leave, 100, 20)), "chunks, overlap 20")

4 chunks, no overlap
5 chunks, overlap 20


### Way 3: Sentence-based chunking

**Concept:** cut at the end of every sentence (`.`  `?`  `!`). Each chunk is a complete sentence.

**Good:** easy to read, nothing is cut in the middle. **Bad:** sentence length changes a lot, and one sentence may not have enough meaning alone ("It costs 20 days.").

**Use it for:** translation, summaries, sentiment (tasks that follow the flow of the language).

In [41]:
import re

def chunk_by_sentence(text):
    sentences = re.split(r"(?<=[.!?])\s+|\n\n", text.strip())       # cut after . ? ! or at a blank line
    return [s for s in sentences if s]

laptop = Path("company_docs/laptop_rules.txt").read_text(encoding="utf-8")
for chunk in chunk_by_sentence(laptop):
    print("-", chunk)

- Laptop Rules
- Every new employee gets one company laptop on the first day.
- You must return it on your last day.
- Do not install unknown software on the company laptop.
- Ask the IT team first.
- If your laptop is lost or stolen, tell the IT team within 24 hours.
- The IT team will lock the laptop from far away.
- Change your laptop password every 90 days.
- Never share your password with anyone.


**✏️ Quick Check:** How many sentence chunks are in `laptop`? (The title counts as one.)

In [42]:
# your code here

**Answer:** 9.

In [43]:
print(len(chunk_by_sentence(laptop)))

9


### Way 4: Paragraph-based chunking

**Concept:** our documents have **one idea per paragraph**, with a **blank line** between paragraphs. We cut at every blank line (`"\n\n"`).

- `strip()` removes spaces and new lines at the start and end.
- We skip very short parts (less than 30 characters). These are titles, like *"Leave Policy"*.

**Good:** one complete idea per chunk. **Bad:** some paragraphs may be very long.

**Use it for:** documents that already have one idea per paragraph, like policies, FAQs and emails (our company docs).

In [44]:
def chunk_by_paragraph(text):
    parts = [p.strip() for p in text.split("\n\n")]     # a blank line = a new paragraph
    return [p for p in parts if len(p) > 30]              # skip very short parts, like titles

for chunk in chunk_by_paragraph(leave):
    print("-", chunk)

- Every employee gets 20 days of paid leave in one year. Unused leave days do not carry over to the next year.
- To take leave, send a request to your manager at least 5 days before. The manager must reply within 2 days.
- Sick leave is separate from paid leave. An employee gets 10 sick days in one year. After 2 sick days in a row, you must show a doctor note.


**✏️ Quick Check:** Which chunk of `leave` has the word **"doctor"**? Use a `for` loop and `if "doctor" in chunk`.

In [45]:
# your code here

**Answer:** the last chunk: *"Sick leave is separate from paid leave..."*

In [46]:
for chunk in chunk_by_paragraph(leave):
    if "doctor" in chunk:
        print(chunk)

Sick leave is separate from paid leave. An employee gets 10 sick days in one year. After 2 sick days in a row, you must show a doctor note.


### Way 5: Recursive chunking

**Concept:** what if a paragraph is **too long**? Try the **big cut** first (paragraph). If a piece is still too long, try a **smaller cut** (sentence). Keep going until every piece fits the size limit.

```
document ──► paragraphs ──► too long? ──► sentences ──► too long? ──► stop
```

**Good:** keeps the structure and also controls the size. **Bad:** a little more code. It works best on documents with clear structure.

**Use it for:** manuals, reports, most real documents. This is a very common default.

In [47]:
def chunk_recursive(text, max_size=120, seps=(r"\n\n", r"(?<=[.!?])\s+")):
    text = text.strip()
    if len(text) <= max_size or not seps:          # small enough (or no cuts left): keep it
        return [text]
    chunks = []
    for part in re.split(seps[0], text):           # use the biggest cut first
        chunks += chunk_recursive(part, max_size, seps[1:])      # then smaller cuts if needed
    return [c for c in chunks if c]

for chunk in chunk_recursive(leave, max_size=120):
    print(len(chunk), "|", chunk)

12 | Leave Policy
108 | Every employee gets 20 days of paid leave in one year. Unused leave days do not carry over to the next year.
107 | To take leave, send a request to your manager at least 5 days before. The manager must reply within 2 days.
39 | Sick leave is separate from paid leave.
42 | An employee gets 10 sick days in one year.
56 | After 2 sick days in a row, you must show a doctor note.


**✏️ Quick Check:** Use `max_size=60`. Do we get more or fewer chunks than with `max_size=120`?

In [48]:
# your code here

**Answer:** more. 6 chunks with 120, 8 chunks with 60. A smaller limit forces more cuts.

In [49]:
print(len(chunk_recursive(leave, 120)), "chunks with max_size=120")
print(len(chunk_recursive(leave, 60)), "chunks with max_size=60")

6 chunks with max_size=120
8 chunks with max_size=60


### Way 6: Semantic chunking

**Concept:** cut where the **meaning changes**. Steps:

1. Split the text into sentences.
2. Make an embedding for each sentence (Day 29).
3. Compare each sentence with the one before it (cosine similarity, Day 24).
4. If the similarity is **low**, the topic changed: start a **new chunk**.

**Good:** each chunk is about one idea, very good for search. **Bad:** slower and costs more, because every sentence needs an embedding before we even start.

**Use it for:** legal or medical documents, where accuracy is more important than speed.

Below we use **real embeddings** from OpenRouter (the same code as Day 24 and Day 29). First, the setup:

In [50]:
import os
import numpy as np
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

load_dotenv()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_KEY"))

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)
EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)

def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    return [item.embedding for item in r.data]                    # one vector per text

def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))     # cosine similarity (Day 24)

Now six sentences: three about **leave**, then three about the **laptop**. Let us see the similarity between each sentence and the one before it.

In [ ]:
sentences = [
    "Employees get 20 days of paid leave in one year.",
    "You must ask your manager 5 days before taking leave.",
    "Sick days are separate from paid leave.",
    "The laptop is given to you on the first day.",
    "Tell the IT team if your laptop is lost.",
    "Never share your laptop password.",
]

vectors = embed(sentences)                                        # 6 sentences = 6 vectors
scores = [similarity(vectors[i - 1], vectors[i]) for i in range(1, len(vectors))]

for i, score in enumerate(scores, start=1):
    print(f"{score:.2f}  {sentences[i - 1][:30]:30} → {sentences[i][:30]}")

Look at the numbers. The score is **low** at the place where the topic changes (sentence 3 → 4). Inside one topic the scores are higher. We choose a **threshold** between them. Here we use **0.25**. A score below it means a new chunk.

In [ ]:
THRESHOLD = 0.25                       # below this = the topic changed

semantic_chunks = [[sentences[0]]]
for sentence, score in zip(sentences[1:], scores):
    if score < THRESHOLD:              # low similarity = the topic changed
        semantic_chunks.append([])     # start a new chunk
    semantic_chunks[-1].append(sentence)

for chunk in semantic_chunks:
    print("-", " ".join(chunk))

**✏️ Quick Check:** How many chunks do we get? Which sentence starts the second chunk?

In [ ]:
# your code here

**Answer:** 2 chunks. The second chunk starts at *"The laptop is given to you on the first day."* because its score is the only one below 0.25. Leave and laptop are different topics. *(The exact numbers can change a little with the model. If your cut is in a different place, change the threshold after you look at the scores.)*

In [ ]:
print(len(semantic_chunks), "chunks")
print(semantic_chunks[1][0])

### Way 7: Hierarchical chunking

**Concept:** keep a **tree**: a big **parent** (the section) and small **children** (the paragraphs). We search with the small children (sharp match). When a child matches, we send its **parent** to the LLM (more context).

```
parent: "Leave Policy" (whole section)
   ├─ child: "Every employee gets 20 days..."
   ├─ child: "To take leave, send a request..."
   └─ child: "Sick leave is separate..."
```

**Good:** sharp search and full context. **Bad:** we must keep the link between parent and child.

**Use it for:** contracts, financial reports, technical specifications.

In [ ]:
parent = {"id": "leave_policy.txt", "text": leave}
children = [
    {"id": f"leave_policy.txt-{i}", "parent_id": parent["id"], "text": text}
    for i, text in enumerate(chunk_by_paragraph(leave))
]

match = children[2]                                  # imagine this child matched the question
print("Matched child:", match["text"][:50], "...")
print("Send to the LLM:", parent["text"][:50], "...")        # the parent has the full context

**✏️ Quick Check:** In this tree, which chunk do we use to **search**, and which to **answer**?

**Answer:** search with the **child** (small and sharp). Answer with the **parent** (big and full of context).

### Way 8: Contextual chunking

**Concept:** add **extra facts** (metadata) to each chunk: the heading, the file name, the date. Many chunks look the same without it. A chunk like *"Late claims are not paid."* does not say **what** is late.

**Good:** better search, and we can show the source (Day 32). **Bad:** a little more work when we prepare the chunks.

**Use it for:** company, legal or compliance documents, where we must show where an answer came from.

In [ ]:
expense = Path("company_docs/expense_policy.txt").read_text(encoding="utf-8")
title = expense.split("\n")[0]                       # the first line is the heading

for text in chunk_by_paragraph(expense)[:2]:
    chunk = {"source": "expense_policy.txt", "title": title, "text": f"[{title}] {text}"}
    print(chunk)

**✏️ Quick Check:** What extra facts does each chunk carry now?

**Answer:** the `source` (file name) and the `title` (heading). The heading is also added in front of the text.

### Way 9: Topic-based chunking

**Concept:** group the text by **topic**. A long text can jump between topics. We put all sentences of one topic in one chunk. Tools for this: topic models (like LDA) or clustering the embeddings.

*Small example:* a meeting note has these sentences:

| Sentence | Topic |
|---|---|
| "The budget is Rs 5 million." | money |
| "We hire 3 new engineers." | people |
| "Cost must stay below Rs 5 million." | money |

Topic-based chunking gives **2 chunks**: one for *money* (sentences 1 and 3), one for *people* (sentence 2). Semantic chunking only cuts when the topic changes, so it would give 3 chunks.

**Use it for:** long reports or articles that move between subjects.

**✏️ Quick Check:** What is the difference between semantic and topic-based chunking?

**Answer:** semantic chunking cuts when the topic **changes** (neighbours only). Topic-based chunking **groups** all sentences of the same topic, even if they are far apart.

### Way 10: Modality-specific chunking

**Concept:** a document may have **text, tables and images**. Each one needs its **own** way of cutting.

| Content | How to chunk |
|---|---|
| Text | By sentence or by paragraph |
| Table | One chunk **per row**, with the column names added |
| Image | Use the caption (or a description) as the text |

If we cut a table like normal text, a row like `Laptop | 150000` loses its column names. We must add them back.

**Use it for:** PDFs and web pages that mix text, tables and images, like a bank statement or a product page.

In [ ]:
table = """Item | Limit | Per
Hotel | 15000 | night
Food | 2000 | day"""

header, *rows = [line.split(" | ") for line in table.split("\n")]
for row in rows:
    print(". ".join(f"{h}: {v}" for h, v in zip(header, row)))

**✏️ Quick Check:** Why do we add the column names to each row?

**Answer:** so every row can be read alone. `15000` means nothing, but `Limit: 15000. Per: night` does.

### Way 11: AI-driven chunking

**Concept:** we ask an **LLM** to read the document and choose where to cut. It can cut a dense section into more pieces and keep a light section together.

**Good:** chunks keep full ideas. **Bad:** slow and expensive (every document goes through an LLM).

**Use it for:** a few high-value documents, like contracts, where one mistake costs a lot.

Below is a **real call** to the LLM (same `client` and `MODEL` as above, Day 24). The text is **one paragraph** with two topics, so paragraph chunking cannot help. We tell the LLM the rules in the **system** message and ask for JSON.

In [ ]:
import json

text = ("Employees get 20 days of paid leave in one year. You must ask your manager 5 days before taking leave. "
        "The laptop is given to you on the first day. Tell the IT team if your laptop is lost. "
        "Never share your laptop password.")

SYSTEM = """You split text into chunks for a search system.
Rules:
1. Put all sentences about the SAME topic in ONE chunk.
2. Start a new chunk only when the topic changes.
3. Do not change, remove or add any word.
Reply with JSON only: a list of strings."""

reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}],
    temperature=0,
)
answer = reply.choices[0].message.content
print(answer)

The LLM gives us **text**. We turn it into a Python list with `json.loads` (Day 24). The model may put the JSON inside a code fence, so we remove it first.

In [ ]:
def parse_json(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

ai_chunks = parse_json(answer)
for chunk in ai_chunks:
    print("-", chunk)

**✏️ Quick Check:** Run the two cells above again. Is the answer **exactly the same** every time? What does this tell you?

**Answer:** not always. Sometimes we get 2 chunks, sometimes 3, even with `temperature=0`. An LLM can change its cuts between runs. So **always look at the chunks** (and check that no word was changed). This is one more reason AI-driven chunking is only for a few important documents.

### Way 12: Agentic chunking

**Concept:** an **AI agent** looks at the document and at what the user wants to know. It **picks a way** for each part, and can mix ways.

*Example:* a hospital report. The agent splits it into *history*, *lab results* and *doctor notes*. It uses semantic chunking on the notes, row chunking on the lab table, and adds the date and doctor ID to every chunk.

**Good:** the best fit for each part. **Bad:** the most complex and the most expensive.

**Use it for:** many different documents that need different treatment.

> **Late chunking:** a newer idea. We first embed the **whole** document, then cut the embeddings into chunks. Each chunk then keeps the context of the full document.

**✏️ Quick Check:** Which is more expensive: sentence chunking or AI-driven chunking? Why?

**Answer:** AI-driven. Every document goes through an LLM, and an LLM call costs tokens (Day 24). Sentence chunking is just a few lines of Python.

## Step 4: How to pick a way

Ask these questions **in order**:

1. **Does the text have clear parts?** (headings, paragraphs) → **Paragraph** or **Recursive**.
2. **No clear parts?** (chat, transcript, story) → **Sliding window**.
3. **Is accuracy worth the extra cost?** (legal, medical) → **Semantic**, or **AI-driven** for the most important files.
4. **Do small chunks lose context?** → **Hierarchical** (send the parent) or **Contextual** (add the heading).
5. **Tables or images inside?** → **Modality-specific**.
6. **Many different document types?** → **Agentic**.
7. **Just need something fast to start?** → **Fixed-size**.

| Your situation | Pick |
|---|---|
| Fast and simple | Fixed-size |
| Needs a natural flow of language | Sentence |
| Clear headings and paragraphs (our company docs) | Paragraph / Recursive |
| Long contract with sections | Hierarchical |
| Must show the source | Contextual |
| Chat log, transcript | Sliding window |
| Long report with many subjects | Topic-based / Semantic |
| Tables and images | Modality-specific |
| Very important, complex files | AI-driven |
| Many kinds of files | Agentic |

**Rules to remember**

- Most real projects start with **Recursive** or **Paragraph**. Change only if the answers are bad.
- A chunk is usually **200–400 tokens**. With overlap, use **20–50%**.
- **Test with real questions.** The best way is the one that finds the right chunk for your users' questions.
- Cost goes up as you move down the table: fixed-size is cheapest, agentic is the most expensive.

**✏️ Quick Check:** What would you pick for each? (a) a 200-page contract with sections and sub-sections, (b) a WhatsApp chat export with no headings, (c) our `company_docs` folder.

**Answer:** (a) **Hierarchical**: sections and sub-sections form a tree. (b) **Sliding window**: no structure, so overlap keeps the context. (c) **Paragraph** or **Recursive**: one idea per paragraph.

## Step 5: All documents → chunks

Now we chunk **every** document by paragraph. Each chunk keeps 3 things:

| Key | Example | Why |
|---|---|---|
| `id` | `"leave_policy.txt-1"` | A unique name. Chroma needs it (Day 29). |
| `source` | `"leave_policy.txt"` | Where it came from. We show it to the user on Day 32. |
| `text` | `"To take leave, send a request..."` | The text we embed. |

In [ ]:
def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({"id": f"{doc['source']}-{i}", "source": doc["source"], "text": text})
    return chunks

chunks = chunk_documents(docs)
print(len(chunks), "chunks")

In [ ]:
import pandas as pd

pd.DataFrame(chunks)          # one row per chunk: easy to read and check

**Always look at your chunks** before you embed them. Bad chunks = bad answers on Day 31.

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone. We use this code again on Day 31.

In [ ]:
from pathlib import Path
import re

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):              # every .txt file in the folder
        docs.append({"source": path.name, "text": path.read_text(encoding="utf-8")})
    return docs

def chunk_by_size(text, size=100, overlap=0):                     # way 1 and 2
    step = size - overlap
    return [text[start:start + size] for start in range(0, len(text), step)]

def chunk_by_sentence(text):                                      # way 3
    return [s for s in re.split(r"(?<=[.!?])\s+|\n\n", text.strip()) if s]

def chunk_by_paragraph(text):                                     # way 4
    parts = [p.strip() for p in text.split("\n\n")]              # a blank line = a new paragraph
    return [p for p in parts if len(p) > 30]                      # skip very short parts, like titles

def chunk_recursive(text, max_size=120, seps=(r"\n\n", r"(?<=[.!?])\s+")):     # way 5
    text = text.strip()
    if len(text) <= max_size or not seps:
        return [text]
    chunks = []
    for part in re.split(seps[0], text):
        chunks += chunk_recursive(part, max_size, seps[1:])
    return [c for c in chunks if c]

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({"id": f"{doc['source']}-{i}", "source": doc["source"], "text": text})
    return chunks

docs = load_documents("company_docs")
chunks = chunk_documents(docs)

print(len(docs), "documents →", len(chunks), "chunks")
for c in chunks[:3]:
    print(c["id"], "|", c["text"][:60])

### Key takeaways

- **Load** files with `Path(...).glob("*.txt")` and `read_text()`. Keep the file name as the **source**.
- **Chunk** = a small part of a document. Goal: **one idea per chunk**.
- Too big = a blurry vector and many tokens. Too small = the meaning is lost.
- The ways: **fixed-size, sliding window, sentence, paragraph, recursive, semantic, hierarchical, contextual, topic-based, modality-specific, AI-driven, agentic**.
- Pick by the document: clear parts → paragraph or recursive. No parts → sliding window. Accuracy first → semantic. Tables → modality-specific.
- Start simple (**recursive** or **paragraph**), then **test with real questions**.
- Every chunk keeps an `id`, a `source` and its `text`. **Always look at your chunks.**

---
# 🏠 Home Problem

The company has a new document: **`remote_work.txt`** (in this folder).

1. Read `remote_work.txt` and print it.
2. Cut it **by size** (`size=120, overlap=30`). Print the chunks. Find one cut word.
3. Cut it **by sentence**. Print the chunks.
4. Cut it **by paragraph**. Print the chunks.
5. Copy `remote_work.txt` into `company_docs`. Run `load_documents` and `chunk_documents` again. How many chunks are there now?
6. In 1 line: which way of cutting is better for `remote_work.txt`? Why?

**Project piece (independent build time):** make a folder `my_docs` for **your own** project. Write 2–3 `.txt` files with one idea per paragraph. Load and chunk them, look at the chunks with `pd.DataFrame`, and write 1 line: which way of chunking did you pick, and why?